# 05 — Context engineering

The last two modules added hands: tools, a loop, a coding agent. Enterprise pilots rarely die because someone forgot `write_file`. They die because **the list fills up**. Every turn is more system text, more schemas, more observations. Cost climbs. The model gets worse the longer it works.

Context engineering is the job of deciding **what is allowed onto that list**.

Module 04 named skills and `map.md` and said they belong here. Cursor and Claude Code do not put every playbook in the system prompt. They keep a short map, and they load one file when they need it. That is **progressive disclosure**. When the list is still too long, they summarise and drop the raw turns. That is **compaction**.

Today you will measure both. Same question, two ways to fill `messages`. The number that matters is `usage.prompt_tokens`.


## 1. Learn

```
00–02  the object, the decision, the tool call
03     the loop
04     the loop on files
05     you are here — the list is a budget
11–12  retrieval (more things that want a seat on the list)
15     evals and cost, after you can see the bill
```

Seven things compete for the same window. Same seven boxes as the slides. Not all of them should ride along on every turn.

| # | Input | What it is | In this notebook |
|---|---|---|---|
| 1 | Instructions | Behaviour and rules | The system message. Skills like `shop.md` are extra rules you load on demand. |
| 2 | User prompt | The immediate request | This question. |
| 3 | Available tools | Every schema, every turn | The `tools=` list. |
| 4 | Structured output | The shape you require back | Named here; we run it in module 15. |
| 5 | State and history | This conversation, so far | The `messages` list, including tool results. |
| 6 | Long-term memory | What survives across sessions | Not today. Module 08's session is the in-run version. |
| 7 | Retrieved information | Documents, databases, APIs | Later, modules 11–12. |

Isolation (a specialist agent that never sees the other specialist's prompt) is a context technique, not an org chart. Module 13 will use it that way.

Two moves today:

1. **Progressive disclosure.** System prompt = the map only. `read_skill` loads one playbook when the model asks. Compare that to stuffing every playbook up front.
2. **Compaction.** After a few observations, replace the raw turns with a short summary. The facts stay. The dumps go.

If the room is behind, keep (1) and cut (2).

The skill files are already on disk under `modules/05_context_engineering/skills/`. We will not generate them in a cell. Open them if you want. They are short on purpose.


## 2. Do

### Load the environment and the files


In [1]:
from pathlib import Path
import csv
import json
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
price_in = float(os.environ["PRICE_INPUT_PER_MILLION"])
price_out = float(os.environ["PRICE_OUTPUT_PER_MILLION"])
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing."

client = OpenAI()
SKILLS = ROOT / "modules" / "05_context_engineering" / "skills"
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("skills:", sorted(p.name for p in SKILLS.glob("*.md")))


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
skills: ['compact.md', 'map.md', 'shop.md', 'travel.md']


### Read the map yourself

This is what should sit in the system prompt. A table. Not the whole library.


In [2]:
def read_skill(name):
    allowed = {"map", "shop", "travel", "compact"}
    if name not in allowed:
        return "unknown skill"
    return (SKILLS / (name + ".md")).read_text()


print(read_skill("map"))
print("map chars:", len(read_skill("map")))
print("all playbooks chars:", sum(len(read_skill(n)) for n in ("map", "shop", "travel", "compact")))


# Skill map

Read this first. Load one skill with read_skill when you need the details. Do not guess the playbook.

| Skill | When to load |
|---|---|
| shop | Customer invoices or support reps (Helena, Puja) |
| travel | Flights between cities, or a fun fact about a city |

If the list of turns is getting long, load compact and summarise.

map chars: 342
all playbooks chars: 7655


The map is small. The four files together are several times that. Every request that stuffs all four pays for shop, travel, and compact even when the question is only Helena.

### The action tools (the hands from 01 and 02)

Same tiny shop. Same CSVs. Call them yourself. The model is not here yet.


In [3]:
SHOP = {
    "helena": {"invoices": 7, "rep": "Steve Johnson"},
    "puja": {"invoices": 6, "rep": "Jane Peacock"},
}

facts_rows = list(csv.DictReader(open(ROOT / "data" / "fun_facts.csv")))
flight_rows = list(csv.DictReader(open(ROOT / "data" / "flight_data.csv")))


def lookup_count(who):
    row = SHOP.get(who.strip().lower())
    return str(row["invoices"]) if row else "unknown customer"


def lookup_rep(who):
    row = SHOP.get(who.strip().lower())
    return row["rep"] if row else "unknown customer"


def get_fact(city):
    needle = city.strip().lower()
    for row in facts_rows:
        if row["City"].lower() == needle:
            return row["Fun Fact"]
    return "no fact for that city"


def get_flight(from_city, to_city):
    found = []
    a = from_city.strip().lower()
    b = to_city.strip().lower()
    for row in flight_rows:
        if row["from_city"].lower() == a and row["to_city"].lower() == b:
            found.append(row["price"] + " dollars, " + row["duration"] + " minutes")
    return "; ".join(found) if found else "no flight found"


print("helena invoices:", lookup_count("helena"))
print("amsterdam fact: ", get_fact("Amsterdam")[:60] + "...")


helena invoices: 7
amsterdam fact:  Amsterdam has more bicycles than people....


### Two system prompts, one question

The question is only about the shop. The fat prompt still carries travel and compact. The thin prompt carries the map and a rule: load a skill if you need it.


In [4]:
QUESTION = "How many invoices does Helena have?"

FAT_SYSTEM = (
    "You answer with tools. All playbooks are below. Do not invent numbers.\n\n"
    + read_skill("map")
    + "\n\n"
    + read_skill("shop")
    + "\n\n"
    + read_skill("travel")
    + "\n\n"
    + read_skill("compact")
)

THIN_SYSTEM = (
    "You answer with tools. Start from the map. "
    "If you need a playbook, call read_skill with shop, travel, or compact. "
    "Do not invent numbers.\n\n"
    + read_skill("map")
)

print("fat system chars: ", len(FAT_SYSTEM))
print("thin system chars:", len(THIN_SYSTEM))


fat system chars:  7733
thin system chars: 480


### Same tools list for both

The schemas ride along either way. What we are measuring is the playbooks, not the five names.

`schema()` below is a small factory: one line per tool instead of five hand-written JSON blocks like module 03 used. Same shape, less repetition.

In [5]:
def schema(tool_name, description, **props):
    return {"type": "function", "function": {
        "name": tool_name, "description": description,
        "parameters": {"type": "object", "properties": props, "required": list(props)}}}

tools = [
    schema("read_skill", "Load one playbook: map, shop, travel, or compact.", name={"type": "string"}),
    schema("lookup_count", "Invoice count for helena or puja.", who={"type": "string"}),
    schema("lookup_rep", "Support rep for helena or puja.", who={"type": "string"}),
    schema("get_fact", "Fun fact about a city.", city={"type": "string"}),
    schema("get_flight", "Flight price and duration between two cities.", from_city={"type": "string"}, to_city={"type": "string"}),
]


### The cell that matters: one create each, compare `prompt_tokens`

Do not run the tools yet. We only care what the **first** request paid to send.


In [6]:
def first_call(system):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": QUESTION},
        ],
        tools=tools,
        max_completion_tokens=128,
        reasoning_effort="none",
    )
    return response


fat = first_call(FAT_SYSTEM)
thin = first_call(THIN_SYSTEM)

print("fat  prompt_tokens:", fat.usage.prompt_tokens)
print("thin prompt_tokens:", thin.usage.prompt_tokens)
print("saved:             ", fat.usage.prompt_tokens - thin.usage.prompt_tokens)
print()
print("fat asked: ", [c.function.name for c in (fat.choices[0].message.tool_calls or [])])
print("thin asked:", [c.function.name for c in (thin.choices[0].message.tool_calls or [])])


fat  prompt_tokens: 2092
thin prompt_tokens: 355
saved:              1737

fat asked:  ['read_skill']
thin asked: ['lookup_count']


The thin call should be cheaper on the way in. That part is the budget.

Look at **what each model asked for**. Stuffing the four playbooks into the fat prompt did not make it skip the load. It asked for `read_skill` — a playbook it is already carrying. The thin prompt, with only the map, is not guaranteed to skip the load either, but it often does: sometimes it goes straight to `lookup_count`, sometimes it loads `shop` first. Either way, extra context did not just cost more. It made the fat model ask for a playbook it was already carrying.

A hundred users, every turn, all day, is that `saved` number times the room. Same `usage` object as module 00.

### Finish the thin path, so you see a load when it is needed

Official loop, cap of 6. Print each tool name. If thin goes straight to `lookup_count`, that is the map doing its job. If it loads `shop` first, that is disclosure working on a later turn instead of the first. Either way, the answer comes from a tool result, not a guess.

In [7]:
def run_call(call):
    args = json.loads(call.function.arguments or "{}")
    name = call.function.name
    if name == "read_skill":
        return read_skill(args.get("name", ""))
    if name == "lookup_count":
        return lookup_count(args.get("who", ""))
    if name == "lookup_rep":
        return lookup_rep(args.get("who", ""))
    if name == "get_fact":
        return get_fact(args.get("city", ""))
    if name == "get_flight":
        return get_flight(args.get("from_city", ""), args.get("to_city", ""))
    return "unknown tool"


messages = [
    {"role": "system", "content": THIN_SYSTEM},
    {"role": "user", "content": QUESTION},
]
names = []
for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=128,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
    if not message.tool_calls:
        print(message.content)
        break
    messages.append(message)
    for call in message.tool_calls:
        result = run_call(call)
        names.append(call.function.name)
        print(call.function.name, "->", str(result)[:80].replace("\n", " / "))
        messages.append({"role": "tool", "tool_call_id": call.id, "content": str(result)})

print("tools in order:", names)


--- turn 1 finish_reason: tool_calls ---
lookup_count -> 7
--- turn 2 finish_reason: stop ---
Helena has **7** invoices.
tools in order: ['lookup_count']


The list now holds the system prompt, the question, a skill file, and tool results. Imagine twenty more observations. Retrieval (module 11) will add documents. That is how the window dies.

Compaction: ask the model for a short summary, then **replace** the middle of `messages` with that summary. Keep the system prompt and the latest user need. Drop the raw dumps.

First, how big is the list now, in prompt tokens? The cell below is not a real question — it appends one throwaway turn ("Say ready") only to trigger a fresh `create()` call, so `usage.prompt_tokens` reports what the whole list already costs before we touch it.

In [8]:
probe = client.chat.completions.create(
    model=model,
    messages=messages + [{"role": "user", "content": "Say ready."}],
    max_completion_tokens=16,
    reasoning_effort="none",
)
print("turns on the list:", len(messages))
print("prompt_tokens now:", probe.usage.prompt_tokens)


turns on the list: 4
prompt_tokens now: 172


Ask for a summary using `compact.md` as the instruction. Then build a shorter list by hand so you can see what stayed and what went.

The summariser needs one block of text, not a list of dicts, so `turn_line` first flattens each entry in `messages` into a single line before it goes into the prompt.

In [9]:
def turn_line(t):
    if isinstance(t, dict):
        role = t["role"]
        content = t.get("content") or ""
    else:
        role = t.role
        content = t.content or ""
        if t.tool_calls:
            content = "tool_calls"
    return role + ": " + content[:300]


transcript = "\n".join(turn_line(t) for t in messages)
compact_ask = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": read_skill("compact")},
        {
            "role": "user",
            "content": "Summarise this conversation. Facts and goal only.\n\n" + transcript,
        },
    ],
    max_completion_tokens=160,
    reasoning_effort="none",
)
summary = compact_ask.choices[0].message.content
print(summary)


Goal: Determine how many invoices Helena has.

Facts already found: Helena has 7 invoices.

What is still open: None.

What to drop: The tool call/trace text (only the final number is needed).


Now build the compacted list by hand: `THIN_SYSTEM`, the summary in place of the raw turns, and the same question again in different words. Asking again is the test — if the fact survived compaction, the model should answer straight from the summary. It should not need to call `lookup_count` a second time.

Notice `tools=` is not even passed to this `create()`. That is on purpose: if compaction worked, this turn does not need hands at all, and the request is cheaper for it, not just shorter.

In [10]:
compacted = [
    {"role": "system", "content": THIN_SYSTEM},
    {"role": "user", "content": "Conversation so far:\n" + summary},
    {"role": "user", "content": "What is Helena's invoice count? One number."},
]

after = client.chat.completions.create(
    model=model,
    messages=compacted,
    max_completion_tokens=32,
    reasoning_effort="none",
)
print("turns before:", len(messages), " turns after:", len(compacted))
print("prompt_tokens after compact:", after.usage.prompt_tokens)
print("answer:", after.choices[0].message.content)


turns before: 4  turns after: 3
prompt_tokens after compact: 190
answer: 7


If the number is still 7, the summary carried the fact and we no longer pay for the skill file dump or the tool traces. If it forgot, the compact was too thin — that is the other failure mode. Compaction is a trade, not a free win.

Isolation, again in one line: a shop-only agent whose system prompt never contained `travel.md` cannot leak flight instructions into a customer answer. That is context, not hierarchy.


## 3. Observe

Put the two first-request sizes next to the compacted size. Dollars on the way in, using the prices from `.env`.


In [11]:
def dollars(prompt_tokens):
    return prompt_tokens / 1_000_000 * price_in


print(f"fat first request:    {fat.usage.prompt_tokens:5d} tokens  ${dollars(fat.usage.prompt_tokens):.6f}")
print(f"thin first request:   {thin.usage.prompt_tokens:5d} tokens  ${dollars(thin.usage.prompt_tokens):.6f}")
print(f"after compact:        {after.usage.prompt_tokens:5d} tokens  ${dollars(after.usage.prompt_tokens):.6f}")


fat first request:     2092 tokens  $0.000418
thin first request:     355 tokens  $0.000071
after compact:          165 tokens  $0.000033


The thin first request is the one you want on every turn of a long day. Compaction is for when the list has already grown. Retrieval will make this worse unless you disclose there too.


## 4. Challenge

Same measurement, new question. Do **not** stuff the travel playbook unless the model asks for it.

> Give me a fun fact about Amsterdam.

Build a fat first request (all four skill files in the system prompt) and a thin first request (map only). Do not finish the loop unless you want to. Bind:

- `fat_tokens` — `prompt_tokens` on the fat first `create`
- `thin_tokens` — `prompt_tokens` on the thin first `create`

The check only asks that thin is strictly smaller. It does not score the wording of any fact.


In [ ]:
# fat_tokens, thin_tokens = ...


In [ ]:
assert fat_tokens > thin_tokens, "the thin first request should send fewer prompt tokens"
print("fat_tokens: ", fat_tokens)
print("thin_tokens:", thin_tokens)
print("saved:      ", fat_tokens - thin_tokens)
